# 高次元入力モデルの Sequential BO ベンチマーク

このNotebookでは、robotorchanの高次元入力モデルを同一条件で比較するPhase 11ベンチマークの使い方を確認します。複数seedでBOを反復し、simple regretの平均だけでなくseed間のばらつきも可視化します。

有限候補プールを使うため、ここで比較しているのは主にsurrogateとacquisition評価の差です。連続高次元空間での `optimize_acqf` 自体の難しさは別に評価します。

In [ ]:
from pathlib import Path
import importlib.util
import sys

import matplotlib.pyplot as plt
import pandas as pd

root = Path.cwd()
if not (root / 'benchmarks' / 'high_dimensional_bo.py').exists():
    root = root.parents[1]
benchmark_path = root / 'benchmarks' / 'high_dimensional_bo.py'
spec = importlib.util.spec_from_file_location('high_dimensional_bo', benchmark_path)
benchmark = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = benchmark
spec.loader.exec_module(benchmark)


## 1. 軽量設定で複数seedのBOを実行

まずは `SingleTaskGP`、`PCAGP`、`PLSGP`、`RandomProjectionGP` の4モデルを比較します。Notebookを短時間で確認できるよう、候補数・反復数・MC sample数は小さめにしています。本格比較では値を増やしてください。

In [ ]:
seeds = [0, 1, 2]
results = benchmark.run_repeated_benchmark(
    seeds,
    n_initial=10,
    n_candidates=64,
    input_dim=20,
    latent_dim=4,
    n_iterations=5,
    mc_samples=16,
)
summaries = benchmark.aggregate_results(results)
len(results), len(summaries)


## 2. 生trajectoryと集計結果をDataFrameで確認

生データにはseedが保存されます。モデル比較では平均値だけでなく、同一seedにおけるpairedなtrajectoryも確認できます。

In [ ]:
raw_df = pd.DataFrame([vars(row) for row in results])
summary_df = pd.DataFrame([vars(row) for row in summaries])
display(raw_df.head())
display(summary_df.head())


## 3. simple regret trajectoryを可視化

線はseed平均、帯は25%点から75%点です。simple regretは小さいほど、候補プール内の良い点を早く発見できています。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for model, frame in summary_df.groupby('model'):
    frame = frame.sort_values('iteration')
    line = ax.plot(frame['iteration'], frame['simple_regret_mean'], marker='o', label=model)[0]
    ax.fill_between(
        frame['iteration'],
        frame['simple_regret_q25'],
        frame['simple_regret_q75'],
        alpha=0.15,
        color=line.get_color(),
    )
ax.set_xlabel('BO iteration')
ax.set_ylabel('Simple regret')
ax.set_title('高次元BO: simple regret trajectory')
ax.legend()
ax.grid(alpha=0.2)
plt.show()


## 4. seedごとのtrajectoryを確認

平均が同程度でも、初期設計への依存が大きいモデルがあります。次の図では各seedを個別に描き、結果が特定seedだけに支配されていないか確認します。

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for (model, seed), frame in raw_df.groupby(['model', 'seed']):
    frame = frame.sort_values('iteration')
    ax.plot(frame['iteration'], frame['simple_regret'], alpha=0.45, label=f'{model} / seed={seed}')
ax.set_xlabel('BO iteration')
ax.set_ylabel('Simple regret')
ax.set_title('seed別 simple regret')
ax.legend(fontsize=7, ncol=2)
ax.grid(alpha=0.2)
plt.show()


## 5. CSVとして保存・再読込

長時間のbenchmarkでは毎回再計算せず、raw trajectoryとsummaryを保存して解析する運用が便利です。

In [ ]:
output_dir = root / 'benchmark_results'
raw_path = output_dir / 'high_dimensional_bo_notebook.csv'
summary_path = output_dir / 'high_dimensional_bo_notebook_summary.csv'
benchmark.write_csv(results, raw_path)
benchmark.write_csv(summaries, summary_path)
loaded_summary = pd.read_csv(summary_path)
loaded_summary.head()


## 6. Extendedモデルを比較する場合

AE/VAE、Supervised AE/VAE、Joint系、MAP-SAASも `include_extended=True` で追加できます。Joint/NeuralモデルをBO反復ごとに学習するため計算時間が大きくなります。本格実行時に必要なモデル群だけを比較してください。

```python
extended_results = benchmark.run_repeated_benchmark(
    [0, 1, 2],
    n_initial=12,
    n_candidates=256,
    input_dim=40,
    latent_dim=5,
    n_iterations=10,
    mc_samples=64,
    include_extended=True,
    neural_epochs=20,
    joint_steps=30,
)
```

## 7. モデル選択時の見方

1. 予測benchmarkでRMSE/NLLが極端に悪いモデルを除外します。
2. 学習時間・posterior時間・acquisition評価時間を確認します。
3. 残ったモデルを複数seedのBOで比較し、simple regretの平均とばらつきを確認します。
4. synthetic benchmarkの結果をそのまま実データの順位として扱わず、実データでも同じ比較を行います。

高次元で連続acquisition optimization自体がボトルネックになる場合は、ReducedGPだけではなくREMBO / BAxUS / TuRBOなど探索戦略側の対応も別途検討します。